In [1]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
from transformers.optimization import AdamW
from source.version2.data import trainLoader
from source.version2.model import XLMModel
from source.version2.train import trainModel

In [3]:
exc = ['bias','LayerNorm.bias','LayerNorm.weight']

In [4]:
def train(load, subset, rate):
    torch.cuda.empty_cache()
    train, valid = trainLoader(subset)
    model = XLMModel()
    if load:
        version = '../../model/version1/model_{}.pt'.format(subset)
        weights = torch.load(version, map_location='cpu')
        weights = weights['model_state_dict']
        model.load_state_dict(weights)
    params = list(model.named_parameters())
    groups = []
    groups += [{'params' : [p for n,p in params if not any(ex in n for ex in exc)], 
                'weight_decay':0.01}]
    groups += [{'params' : [p for n,p in params if any(ex in n for ex in exc)], 
                'weight_decay':0.00}]
    model = model.to('cuda:0')
    optimizer = AdamW(groups, lr=rate)
    schedular = ReduceLROnPlateau(optimizer, factor=0.1, min_lr=1e-7)
    trainer = {}
    trainer['subset'] = subset
    trainer['model'] = model
    trainer['train'] = train
    trainer['valid'] = valid
    trainer['loss_fn'] = nn.BCEWithLogitsLoss(reduction='none')
    trainer['optimizer'] = optimizer
    trainer['save'] = '../../model/version2/'
    trainer['epochs'] = 4
    trainer['batch'] = 8
    trainer['schedular'] = schedular
    trainModel(**trainer)
    model = model.cpu()
    del model
    torch.cuda.empty_cache()
    return None

In [5]:
train(True, 0, 1e-6)

 90%|█████████ | 25984/28760 [33:45<03:45, 12.28it/s, train_loss=0.2473]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

100%|██████████| 28760/28760 [40:56<00:00, 11.71it/s, val_loss=0.2678, val_mtrc=0.9271]


In [ ]:
train(True, 1, 1e-6)

 53%|█████▎    | 15136/28760 [19:40<17:29, 12.98it/s, train_loss=0.2581]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

 40%|████      | 11552/28760 [15:00<22:04, 12.99it/s, train_loss=0.2426]

In [ ]:
train(True, 2, 1e-6)

In [ ]:
train(True, 3, 1e-6)

In [ ]:
train(True, 4, 1e-6)